In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
from langchain_openai import ChatOpenAI

In [20]:
from pydantic import BaseModel
from typing import List

In [14]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [21]:
class Queries(BaseModel):
    queries: List[str]

In [25]:
str_res = llm.with_structured_output(Queries)
queries = str_res.invoke("Hi").queries
print(queries)

['Hello! How can I assist you today?', 'What is on your mind?', 'Do you have any questions for me?', 'How can I help you today?']


In [1]:
q = ["a", "b", "c"]
f = ["x", "y", "z"]

In [8]:
exist = [{"query": q_, "find": f_} for q_,f_ in zip(q,f)]

In [12]:
new = {"query": 'c', 'find': 'k'}

In [15]:
map = {item['query']: item for item in exist}
print(map)

{'a': {'query': 'a', 'find': 'x'}, 'b': {'query': 'b', 'find': 'y'}, 'c': {'query': 'c', 'find': 'z'}}


In [16]:
map[new['query']] = new

In [17]:
map.values()

dict_values([{'query': 'a', 'find': 'x'}, {'query': 'b', 'find': 'y'}, {'query': 'c', 'find': 'k'}])

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from langchain_tavily import TavilySearch

In [22]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI 
from app.langgraph.prompts import get_research_prompt

In [23]:
small_llm = ChatOpenAI(model="gpt-4o-mini")

In [45]:
@tool
def search_web(query_obj):
    """
        Search the web for factual information related to the given query.
    
        Args:
            query: The search query to look up on the web.
    
        Returns:
            A dictionary containing:
            - findings: The relevant information extracted from the search result.
            - sources: The URL of the source containing the information.
    """

    tool = TavilySearch(
                    max_results=1,
                    topic="general",)
    result = tool.invoke(query_obj)

    if isinstance(result, dict) and "results" in result and result["results"]:
        data = result["results"][0]
        return {
            "query": query_obj["query"],
            "findings": data.get("content", "No content found."),
            "sources": data.get("url", "No URL found."),
        }
    return {"query": query_obj["query"],
            "findings": "No content found.", 
            "sources": "No URL found."}

tools = [search_web]

In [46]:
llm_with_tool = small_llm.bind_tools(tools)

In [47]:
res = llm_with_tool.invoke(get_research_prompt("Who won last odi world cup"))

In [48]:
q= res.tool_calls[0]['args']

In [42]:
type(search_web.invoke(q))

dict

In [49]:
search_web.invoke(q)

{'query': 'Who won the last ODI World Cup',
 'findings': "Photo by Neelendra Pratap on September 11, 2026. May be an image of cricket and text that says 'Blaa ROHIT IN ODI WC 1575 RUNS 28 MATCHES 60.58 AVERAGE MRA Genius 海创 UP KOHLI IN ODI WC 1795 RUNS 37 MATCHES 59.83 AVERAGE'.\n\nPhoto by Neelendra Pratap on September 11, 2026. May be an image of cricket, poster and text that says 'VE MA olms Guess The Bowler.?'. [...] Video by Neelendra Pratap on September 04, 2026. May be an image of candle, cake and text that says 'ล Happiest birthday kanha ji...... ······'.\n\nPhoto by Neelendra Pratap on August 25, 2026. May be an image of cricket and text that says 'Rahulthako Rahul thakor MOST 50+ SCORES FOR INDIA IN WORLD TEST CHAMPIONSHIP RISHABH PANT 24 RAVINDRA JADEJA 23 YASHASVI JAISWAL 20 SHUBMAN GILL 19 ROHIT SHARMA 17 CHETESHWAR PUJARA 16 RAHUL 16 VIRAT KOHLI 16 AJINKYA RAHANE 12 MAYANK AGARWAL 8'.",
 'sources': 'https://www.instagram.com/p/DVVG0cYAX9g?hl=en'}

In [50]:
tool_calls=[{'name': 'search_web', 'args': {'query': 'last ODI World Cup winner 2023'}, 'id': 'call_1Xvt9pHYKs1aVBI7VaVqy9iK', 'type': 'tool_call'}]

In [53]:
tool_calls[0]['args'].get('query', 'Error')

'last ODI World Cup winner 2023'

In [54]:
results = [
    {
        "query": "What is artificial intelligence?",
        "finding": "Artificial intelligence is the simulation of human intelligence by machines.",
        "sources": "https://example.com/ai"
    },
    {
        "query": "What is machine learning?",
        "finding": "Machine learning is a branch of AI that enables systems to learn from data.",
        "sources": "https://example.com/ml"
    },
    {
        "query": "What is deep learning?",
        "finding": "Deep learning uses neural networks with multiple layers to learn complex patterns.",
        "sources": "https://example.com/deep-learning"
    }
]